In [0]:
from pyspark.sql.functions import expr, when, current_timestamp, lit, col, to_date
from pyspark.sql import functions as F

In [0]:
tabela_tarifas = spark.read.table('`ct-esteira-dados-aviacao`.raw.tarifas_voos')

In [0]:
df_tarifas = (
    tabela_tarifas
    .withColumn('ano', col('ano').cast("int"))
    .withColumn('mes', col('mes').cast("int"))
    .withColumn('assentos', col('assentos').cast("int"))
    .withColumn('tarifa', F.regexp_replace('tarifa', ',', '.').cast("double"))
    .withColumn(
        "data_ref",
        F.to_date(F.concat_ws("-", col('ano'), col('mes'), F.lit('01')))
    )
    
)

In [0]:
display(df_tarifas)

In [0]:
df_tarifas.createOrReplaceTempView("tarifas")
tabela_siglas = spark.read.table('`ct-esteira-dados-aviacao`.trusted.dim_aero_siglas_truested')
tabela_siglas.createOrReplaceTempView("siglas")

In [0]:
cias_aerea = spark.read.table("`ct-esteira-dados-aviacao`.raw.dim_cias_aereas")
cias_aerea.createOrReplaceTempView("cias_aerea")

In [0]:
df_tarifas_final = spark.sql("""
SELECT
  t.ano,
  t.mes,
  ca.empresa_aerea AS empresa,
  s_origem.s_aero_iata AS origem_iata,
  s_destino.s_aero_iata AS destino_iata,
  t.tarifa,
  t.data_ref
FROM tarifas t
JOIN siglas s_origem ON t.origem = s_origem.s_aero_icao
JOIN siglas s_destino ON t.destino = s_destino.s_aero_icao
JOIN cias_aerea ca ON t.empresa = ca.sigla_cia_aerea
WHERE empresa like '%GLO%' or empresa like "%AZU" or empresa like "%TAM"
""")


In [0]:
display(df_tarifas_final)

In [0]:
df_tarifas_final = df_tarifas_final.withColumn("load_data", current_timestamp())

In [0]:
df_tarifas_final.write\
    .format("delta") \
    .option("overwriteSchema", "true") \
    .mode("overwrite") \
    .partitionBy("ano", "mes") \
    .saveAsTable("`ct-esteira-dados-aviacao`.business.tarifas_voos")